In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import tensorflow as tf 
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Load the preprocessed data
data = pd.read_csv('../Clustering_Analysis/dataset_with_clusters.csv')
data.head()

In [ ]:
# Splitting the data set into training and testing sets\

tf.random.set_seed(42)
np.random.seed(42)

X_train, X_test, y_train, y_test = train_test_split(data.drop('Churn', axis=1), data['Churn'], test_size=0.2, random_state=42, stratify=data['Churn'])

In [ ]:
# Checking shape and distribution
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
print("Training set class distribution:\n", y_train.value_counts(normalize=True))
print("Testing set class distribution:\n", y_test.value_counts(normalize=True))

In [ ]:
# Scaling the features
scaler = StandardScaler()

number_cols = ['tenure', 'MonthlyCharges']

# Fit the scaler on the training data and transform both training and testing data
X_train[number_cols] = scaler.fit_transform(X_train[number_cols])
X_test[number_cols] = scaler.transform(X_test[number_cols])

print(X_train.head(),"\n")
print(X_train[['tenure', 'MonthlyCharges']].describe())

In [ ]:
# Creating an ANN Model
model = Sequential([
    Dense(128, activation="elu", input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.25),

    Dense(64, activation="elu"),
    BatchNormalization(),
    Dropout(0.20),

    Dense(16, activation="elu"),
    Dropout(0.15),

    Dense(1, activation="sigmoid")
])

In [ ]:
# Build class weights to handle class imbalance
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))

In [ ]:
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()

In [ ]:
# Train the model with early stopping

early_stop = EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True)

history = model.fit(
X_train, y_train,
validation_split=0.2,
epochs=60,
batch_size=32,
callbacks=[early_stop],
verbose=1
)

In [ ]:
# Training accuray and Validation accuracy
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show() 

In [ ]:
# Loss curve
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
test_loss, test_accuracy, test_auc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test AUC: {test_auc:.4f}")

y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", cm)

In [ ]:
# Plotting Seaborn Confusion Matrix
import seaborn as sns

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC Curve

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, color='blue', label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show() 